# Co (HCP) – EMTOFlow example

This notebook demonstrates how to:

- Load the Co HCP YAML configuration (`examples/Co_hcp/Co_hcp_example.yaml`)
- Validate the configuration using the **EMTOFlow** library
- Run the **prepare-only** workflow via the Python API (no EMTO executables needed)
- Understand how the same config can be used with the **CLI** (`emtoflow-opt`)

## Pre-requisites

Before running this notebook, make sure you have:

- Installed this repository as a package (from the repo root):

```bash
pip install -e .

In [1]:
from pathlib import Path

from emtoflow import (
    OptimizationWorkflow,
    load_and_validate_config,
)

# Path to the Co HCP YAML config
config_path = Path("Co_hcp_example.yaml")
print(config_path.read_text())

output_path: "emto_inputs_Co_hcp"
job_name: "Co_hcp_example"

# Structure from lattice parameters (HCP Co)
cif_file: null
lat: 4
a: 2.51
b: null
c: 4.069
alpha: null
beta: null
gamma: null
sites:
  - position: [0.0, 0.0, 0.0]
    elements: ["Co"]
    concentrations: [1.0]
  - position: [0.3333, 0.6667, 0.5]
    elements: ["Co"]
    concentrations: [1.0]

# Parameter ranges (single-point, auto from structure)
ca_ratios: null
sws_values: null
initial_sws: null
auto_generate: false
ca_step: 0.02
sws_step: 0.05
n_points: 7

# EMTO calculation parameters
dmax: 1.8
magnetic: "P"
user_magnetic_moments: {"Co": 0.0}
functional: "GGA"

# Optimization settings (disabled for minimal example)
optimize_ca: false
optimize_sws: false
optimize_dmax: false
dmax_initial: 2.0
dmax_target_vectors: 100
dmax_vector_tolerance: 15

# K-mesh
nkx: 21
nky: 21
nkz: 21
rescale_k: false

# Execution / workflow
prepare_only: true
run_mode: "local"
prcs: 4
slurm_account: "name_of_project"
slurm_time: "02:00:00"
slurm_

## Understanding the Co HCP configuration

Key points in `Co_hcp_example.yaml`:

- Uses **lattice parameters** (no CIF) to define HCP Co.
- Sets `prepare_only: true`, so the workflow:
  - Builds the EMTO structure.
  - Generates KSTR/SHAPE/KGRN/KFCD input files.
  - **Does not** call any EMTO executables.
- Is suitable as a minimal example that works without an EMTO installation.

Next, we will:

1. Validate the configuration using `load_and_validate_config`.
2. Run the workflow via `OptimizationWorkflow`.

In [4]:
# Load and validate the YAML configuration using EMTOFlow's config parser
config = load_and_validate_config(str(config_path))

# Show a few important fields
{
    "output_path": config.get("output_path"),
    "job_name": config.get("job_name"),
    "prepare_only": config.get("prepare_only"),
    "lat": config.get("lat"),
    "a": config.get("a"),
    "c": config.get("c"),
}

{'output_path': 'emto_inputs_Co_hcp',
 'job_name': 'Co',
 'prepare_only': True,
 'lat': 4,
 'a': 2.51,
 'c': 4.069}

In [5]:
# Run the optimization workflow in prepare-only mode for this config

workflow = OptimizationWorkflow(config=str(config_path))
results = workflow.run()

print("Phases completed:", list(results.keys()))
print("Output written to:", workflow.base_path)


################################################################################
# PREPARE-ONLY MODE: Creating Input Files
################################################################################

Job: Co
Output: emto_inputs_Co_hcp
Mode: Input file generation only (no calculations)
################################################################################

STEP 1: STRUCTURE CREATION
Creating structure from parameters...
✓ Structure created
  Lattice: Hexagonal (type 4)
  Atoms: 2

STEP 2: PARAMETER PREPARATION
Auto-generated c/a ratios around 1.6211: [np.float64(1.5611155378486057), np.float64(1.5811155378486057), np.float64(1.6011155378486057), np.float64(1.6211155378486057), np.float64(1.6411155378486058), np.float64(1.6611155378486058), np.float64(1.6811155378486058)]
Auto-generated SWS values around 2.6151: [np.float64(2.465058608223542), np.float64(2.515058608223542), np.float64(2.5650586082235423), np.float64(2.615058608223542), np.float64(2.665058608223542), np.fl

## Running the same example via CLI

Once `emtoflow` is installed, you can run the *same* configuration directly from the command line
(without using this notebook):

```bash
# From the repository root
emtoflow-opt examples/Co_hcp/Co_hcp_example.yaml